# Lektion 12 - Reduktion af chat-historik med agentens kladdeblok

Denne notesbog demonstrerer, hvordan man styrer kontekst i lange samtaler ved hjælp af Microsoft Agent Framework. Efterhånden som samtalerne vokser, stiger antallet af tokens — og overskrider til sidst modellens kontekstvindue. Vi håndterer dette med et **mønster til kontekstopsummering** og en **agent-kladdeblok** til vedvarende hukommelse.

## Hvad du vil lære:
1. **Hvorfor kontekststyring er vigtigt**: Forståelse af token-grænser og kontekstvinduer
2. **Kontekstbevidste agenter**: At bygge agenter, der styrer deres egen samtalekontekst
3. **Mønster til kontekstopsummering**: Brug af værktøjer til at kondensere samtalehistorik
4. **Agent-kladdeblok**: Vedvarende hukommelse, der overlever kontekstreduktion

## Forudsætninger:
- Azure OpenAI setup med miljøvariabler konfigureret
- Forståelse af grundlæggende agentbegreber fra tidligere lektioner


## Opsætning


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Hvorfor kontekststyring er vigtigt

Hver LLM har et begrænset **kontekstvindue** — det maksimale antal tokens, den kan behandle i en enkelt forespørgsel. Når en samtale med flere runder skrider frem:

- **Token-tællingen vokser lineært** med hver brugermeddelelse og assistentsvar.
- **Prompt-tokens dominerer omkostningerne**, fordi hele historikken sendes igen ved hver runde.
- Til sidst **overskrides kontekstvinduet** i samtalen, og modellen forkorter enten eller fejler.

### Strategier til kontekststyring

| Strategi | Hvordan det virker | Afvejning |
|---|---|---|
| **Forkortelse** | Drop de ældste beskeder | Mister tidlig kontekst |
| **Sammenfatning** | Kondenser ældre beskeder til et resumé | Noget detaljer går tabt, men nøglepunkter bevares |
| **Notatblok / Ekstern hukommelse** | Gem nøglefakta udenfor samtalen | Kræver værktøjskald, men overlever enhver reduktion |

I denne notesbog kombinerer vi **sammenfatning** med et **notatblok-værktøj**, så agenten kan bevare kontinuitet, selv når samtalehistorikken er kondenseret.


## Oprette en kontekstbevidst agent


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Simulering af en Lang Samtale

Lad os gennemgå en samtale med flere omgange for at se, hvordan kontekst akkumuleres. Agenten skal bevare væsentlige detaljer (præferencer, budget, rejsedatoer) på tværs af omgange og demonstrere kontinuitet.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Bemærk hvordan agenten bevarer konteksten fra tidligere runder — den ved noget om Japan, sushi, templer, fotografi, budgettet på $3000, solo rejser og april-tidsrammen. I en kort samtale fungerer dette godt, men efterhånden som samtalen vokser, bliver det dyrt at sende hele historikken igen.

Lad os fortsætte samtalen med flere runder for at se kontekstakkumuleringen:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Mønster for kontekstopsummering

Efterhånden som samtalen vokser, kan vi bruge et **opsummeringsværktøj** til at komprimere den akkumulerede kontekst til et kompakt format. Agenten kalder dette værktøj for at registrere nøglepræferencer, så selv hvis ældre beskeder bliver slettet, bevares den væsentlige information.

Dette mønster er byggesten for en mere sofistikeret historikreduktion:
1. Agenten identificerer nøglefakta fra samtalen
2. Den kalder opsummeringsværktøjet for at gemme dem
3. Ældre beskeder kan fjernes sikkert, fordi opsummeringen fanger det væsentlige

Nedenfor definerer vi et `summarize_preferences` værktøj, som agenten kan kalde for at gemme en kompakt opsummering af, hvad den har lært.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Resumé

I denne lektion lærte du, hvordan man håndterer kontekst i langvarige agent-samtaler ved hjælp af Microsoft Agent Framework:

### Nøglebegreber
- **Kontekstvinduer er begrænsede** — hver token i samtalehistorikken koster penge og tæller med i grænsen.
- **Sammenfattende værktøjer** lader agenten komprimere opsamlet kontekst til kompakte resuméer, hvilket reducerer tokenforbruget samtidig med, at væsentlig information bevares.
- **Agent scratchpads** giver vedvarende ekstern hukommelse, som overlever enhver samtalereduktion.

### Hvad du byggede
- En **kontekstbevidst agent**, der opretholder kontinuitet på tværs af samtaler med flere rulninger
- Et **sammenfattende værktøj** (`summarize_preferences`), der optager nøgleoplysninger om brugeren i et kompakt format
- En **samtale med flere rulninger**, der demonstrerer kontekstbevægelse og håndtering af ændringer

### Anvendelser i den virkelige verden
- **Kundeservicebots**: Husk præferencer på tværs af lange supportsessioner
- **Personlige assistenter**: Følg igangværende projekter uden at skulle forklare kontekst igen
- **Uddannelsesvejledere**: Oprethold elevfremskridt gennem mange interaktioner

### Næste skridt
- Implementer et komplet scratchpad-værktøj med filbaseret vedvarende lagring
- Tilføj automatisk historikafsavning efter sammenfatning
- Kombiner med vektordatabaser for semantisk hukommelsessøgning
- Byg agenter, der kan genoptage samtaler dage senere med fuld kontekst


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
